# Exercice 1 — Robot dans un labyrinthe

**Codage :** chromosome de `L` mouvements. Chaque gène vaut `0` (Haut), `1` (Bas), `2` (Gauche) ou `3` (Droite).

**Convention :** les coordonnées Python commencent à 0. Ainsi, l'entrée du sujet `(1,1)` devient `(0,0)` et la sortie `(10,10)` devient `(9,9)`.

In [23]:
import random

# 0 = mur ; 1 = case libre
LABYRINTHE = [
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 0, 1],
    [1, 1, 0, 0, 0, 1, 0, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 0, 1, 1, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 1, 0, 1, 1, 1, 1]
]

N = len(LABYRINTHE)
ENTREE = (0, 0)
SORTIE = (N - 1, N - 1)

# 0 = Haut, 1 = Bas, 2 = Gauche, 3 = Droite
MOUVEMENTS = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1)
}
NOMS_MOUVEMENTS = {0: 'H', 1: 'B', 2: 'G', 3: 'D'}

TAILLE_CHROMOSOME = 30

## 1) Proposer un codage des individus
Un individu est une séquence de 30 mouvements. Chaque gène représente un déplacement.

In [24]:
def creer_individu():
    return [random.randint(0, 3) for _ in range(TAILLE_CHROMOSOME)]

def creer_population(taille_population):
    return [creer_individu() for _ in range(taille_population)]

## 2) Définir une fonction fitness
On simule le déplacement. Le robot s'arrête s'il rencontre un mur, sort de la grille, revisite une case ou atteint la sortie.

La fonction est à **maximiser** : atteindre la sortie donne une grande récompense. Sinon, une distance de Manhattan plus petite donne un meilleur score.

In [25]:
def simuler(individu):
    position = ENTREE
    visitees = {ENTREE}
    nb_pas = 0

    for gene in individu:
        di, dj = MOUVEMENTS[gene]
        ni = position[0] + di
        nj = position[1] + dj

        # Arrêt si le robot sort de la grille ou rencontre un mur
        if not (0 <= ni < N and 0 <= nj < N):
            break
        if LABYRINTHE[ni][nj] == 0:
            break

        # Arrêt si le robot revisite une case
        if (ni, nj) in visitees:
            break

        position = (ni, nj)
        visitees.add(position)
        nb_pas += 1

        if position == SORTIE:
            break

    return position, nb_pas

def fonction_fitness(individu):
    position, nb_pas = simuler(individu)

    if position == SORTIE:
        return 1000 + (TAILLE_CHROMOSOME - nb_pas)

    distance = (abs(position[0] - SORTIE[0])
                + abs(position[1] - SORTIE[1]))

    return -distance + nb_pas / (TAILLE_CHROMOSOME + 1)

## 3) Proposer les opérateurs génétiques

In [26]:
def selection_tournoi(population, k=3):
    candidats = random.sample(population, k)
    return max(candidats, key=fonction_fitness)

def croisement(parent1, parent2):
    point = random.randint(1, TAILLE_CHROMOSOME - 1)
    enfant1 = parent1[:point] + parent2[point:]
    enfant2 = parent2[:point] + parent1[point:]
    return enfant1, enfant2

def mutation(individu, pm=0.05):
    nouvel_individu = individu[:]
    for i in range(len(nouvel_individu)):
        if random.random() < pm:
            mouvements_possibles = [m for m in range(4) if m != nouvel_individu[i]]
            nouvel_individu[i] = random.choice(mouvements_possibles)
    return nouvel_individu

## 4) Code complet de l'algorithme génétique

In [31]:
def algorithme_genetique(taille_population=200, max_generations=300,
                         pc=0.9, pm=0.05):
    population = creer_population(taille_population)
    meilleure_solution = max(population, key=fonction_fitness)

    for generation in range(max_generations):
        nouvelle_population = [meilleure_solution[:]]  # élitisme

        while len(nouvelle_population) < taille_population:
            parent1 = selection_tournoi(population)
            parent2 = selection_tournoi(population)

            if random.random() < pc:
                enfant1, enfant2 = croisement(parent1, parent2)
            else:
                enfant1, enfant2 = parent1[:], parent2[:]

            enfant1 = mutation(enfant1, pm)
            enfant2 = mutation(enfant2, pm)
            nouvelle_population.extend([enfant1, enfant2])

        population = nouvelle_population[:taille_population]
        candidat = max(population, key=fonction_fitness)

        if fonction_fitness(candidat) > fonction_fitness(meilleure_solution):
            meilleure_solution = candidat[:]

        if simuler(meilleure_solution)[0] == SORTIE:
            print('Sortie trouvée à la génération', generation)
            break

    return meilleure_solution

## Exécution et affichage du résultat

In [28]:
solution = algorithme_genetique()
position, nb_pas = simuler(solution)
chemin = ''.join(NOMS_MOUVEMENTS[g] for g in solution[:nb_pas])

print('Chemin :', chemin)
print('Position finale (coordonnées du sujet) :',
      (position[0] + 1, position[1] + 1))
print('Sortie atteinte :', position == SORTIE)
print('Fitness :', fonction_fitness(solution))

Sortie trouvée à la génération 33
Chemin : BBDDDBBDDBBDBBBDDD
Position finale (coordonnées du sujet) : (10, 10)
Sortie atteinte : True
Fitness : 1012


## Affichage du chemin trouvé sur le labyrinthe
Cette partie affiche le labyrinthe après l'exécution de l'algorithme :
- `█` = mur (`0`)
- `·` = case libre (`1`) non visitée
- `●` = case parcourue par le robot


In [29]:
def afficher_chemin(individu):
    position = ENTREE
    chemin = [ENTREE]
    visitees = {ENTREE}

    for gene in individu:
        di, dj = MOUVEMENTS[gene]
        ni = position[0] + di
        nj = position[1] + dj

        # Arrêt si le robot sort de la grille ou rencontre un mur (0)
        if not (0 <= ni < N and 0 <= nj < N):
            break
        if LABYRINTHE[ni][nj] == 0:
            break

        # Arrêt si le robot revient sur une case déjà visitée
        if (ni, nj) in visitees:
            break

        position = (ni, nj)
        visitees.add(position)
        chemin.append(position)

        if position == SORTIE:
            break

    for i in range(N):
        ligne = []
        for j in range(N):
            if LABYRINTHE[i][j] == 0:
                symbole = '█'       # mur
            elif (i, j) in visitees:
                symbole = '●'       # chemin parcouru
            else:
                symbole = '·'       # case libre
            ligne.append(symbole)
        print(' '.join(ligne))

    print('\nEntrée :', (ENTREE[0] + 1, ENTREE[1] + 1))
    print('Position finale :', (position[0] + 1, position[1] + 1))
    print('Sortie atteinte :', position == SORTIE)

afficher_chemin(solution)

● █ · · █ · · · · ·
● █ · · █ · █ █ █ █
● ● ● ● █ · · · · ·
█ █ █ ● █ · █ · █ ·
· · · ● ● ● █ · █ ·
· · █ █ █ ● █ · █ ·
· █ · · · ● ● · █ ·
· █ · █ · █ ● · · ·
· █ · █ · █ ● █ █ █
· █ · █ · █ ● ● ● ●

Entrée : (1, 1)
Position finale : (10, 10)
Sortie atteinte : True
